# Experiment 1 — Navigation Element Recognition (single model)

Evaluates **one** PaliGemma checkpoint on the UFV Campus dataset: constrained answers (yes/no, open/closed, up/down) plus short **element** questions. Outputs CSV, JSON, and PNG plots under `/content/results/exp1/` for your TFG.

---

## Notebook structure

| § | Topic |
|---|--------|
| **1** | Install dependencies |
| **2** | **Unzip** `images.zip` + `annotations.zip` |
| **3** | **Configuration** — edit **`MODEL_ID`** (only change between runs) and **`HF_TOKEN`** |
| **4** | Imports + load `questions.json` / `exp1_expected_answers.json` |
| **5** | Load model |
| **6** | Metrics helpers + **inference loop** |
| **7** | BERTScore, aggregates, efficiency, **save CSV/JSON** |
| **8** | Numeric matrices + **PNG plots** |

**Categories** (`images/<name>/`)

| Category | Role | Pos. | Neg. |
|----------|------|:-:|:-:|
| `staircase` | Stairs, direction, handrails | 13 | 2 |
| `elevator` | Cabin doors | 8 | 2 |
| `door` | Plain doors | 5 | 2 |
| `door_sign` | Bathroom / exit | 5 | 2 |
| `obstacle` | Blocking objects + safety | 7 | 2 |

**Metrics (state in memoria)**

- **Accuracy (closed)** — *constrained* match (first token + yes/no phrases), not strict string equality.
- **Precision / Recall / F1** — standard TP/FP/TN/FN.
- **`hazard_miss_rate` (`obstacle_detection`)** — `1 − recall`: fraction of **obstacles missed** (model says no obstacle when there is one). Lower is better.
- **`safety_hazard_miss_rate` (`obstacle_safety`, path clear)** — `FP / (FP + TN)` where the positive class is *clear path* (`yes`): among scenes that are **not** safe to treat as clear, how often the model answers *clear* anyway. Lower is better.
- **Keyword column** — binary hit if *any* expected keyword appears in the prediction.
- **BERTScore** — `lang='en'`.

---

## How to run (Colab)

1. **Runtime → GPU** (T4+ for 3B; A100-class for 10B).
2. Upload **`images.zip`** and **`annotations.zip`** to **`/content/`**.
3. Run **§2 Unzip**, then set **§3 Configuration**.
4. **Runtime → Run all** (or run from §4).

**Switch model:** change **`MODEL_ID`** in §3 and re-run from §4 (or Run all).

---

## Output (one flat folder per run)

```
/content/results/exp1/
├── {model_safe}_exp1_results.csv
├── {model_safe}_exp1_aggregate.csv
├── {model_safe}_exp1_efficiency.json
├── {model_safe}_exp1_matrices.json
└── plots/   (heatmaps + accuracy bars)
```


In [ ]:
# Install dependencies
!pip install -q transformers torch Pillow bert-score accelerate

## 2. Unzip dataset

Upload **`images.zip`** and **`annotations.zip`** to **`/content/`** (Colab sidebar).

Expected after extraction:

- `/content/images/{staircase,elevator,door,door_sign,obstacle}/` — `.jpg` files  
- `/content/annotations/questions.json` and `exp1_expected_answers.json`


In [ ]:
import os
import zipfile

CONTENT = '/content'
ARCHIVES = [
    ('images.zip', 'images'),
    ('annotations.zip', 'annotations'),
]

for zip_name, folder_name in ARCHIVES:
    zip_path = os.path.join(CONTENT, zip_name)
    dest = os.path.join(CONTENT, folder_name)
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(
            f"Missing {zip_path!r}. Upload {zip_name} to /content/."
        )
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest)
    print(f'OK: {zip_name} → {dest}/')

for p in (
    os.path.join(CONTENT, 'annotations', 'questions.json'),
    os.path.join(CONTENT, 'annotations', 'exp1_expected_answers.json'),
):
    if not os.path.isfile(p):
        raise FileNotFoundError(f'Expected file missing after unzip: {p}')

for sub in ('staircase', 'elevator', 'door', 'door_sign', 'obstacle'):
    d = os.path.join(CONTENT, 'images', sub)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.lower().endswith('.jpg')])
        print(f'  images/{sub}/: {n} .jpg files')
    else:
        print(f'  WARNING: missing folder {d}')

print('Unzip complete.')


## 3. Configuration

Set **`MODEL_ID`** (HuggingFace checkpoint) and **`HF_TOKEN`** if the model is gated. Paths assume **§2** has run.


In [ ]:
# Configuration - change MODEL_ID to switch models
MODEL_ID = 'google/paligemma2-3b-pt-224'
HF_TOKEN = ''  # Set your HuggingFace token here

# Paths - no changes needed if you uploaded images/ and annotations/ to /content/
IMAGES_PATH = '/content/images'
ANNOTATIONS_PATH = '/content/annotations'
RESULTS_PATH = '/content/results/exp1'

MAX_NEW_TOKENS = 30

print(f'Model: {MODEL_ID}')
print(f'Images path: {IMAGES_PATH}')
print(f'Annotations path: {ANNOTATIONS_PATH}')
print(f'Results path: {RESULTS_PATH}')

## 4. Imports

PyTorch, PaliGemma, pandas, BERTScore. `display` falls back to `print` outside Jupyter/Colab.


In [ ]:
# Imports
try:
    from IPython.display import display
except ImportError:
    display = print

import os
import json
import time
import torch
import numpy as np
import pandas as pd
from PIL import Image
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor
from bert_score import score as bert_score_fn

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 200)
print('Imports OK')

In [ ]:
# Load model and processor
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

load_start = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32
)
model = model.to(device)
model.eval()
model_load_time_s = round(time.time() - load_start, 2)

n_parameters = sum(p.numel() for p in model.parameters())
vram_load_gb = torch.cuda.max_memory_allocated() / (1024**3) if device == 'cuda' else 0
print(f'Model loaded: {MODEL_ID}')
print(f'Parameters: {n_parameters:,} ({n_parameters/1e9:.2f}B)')
print(f'VRAM after load: {vram_load_gb:.2f} GB')
print(f'Load time: {model_load_time_s:.1f}s')

In [ ]:
# Load questions and expected answers
with open(f'{ANNOTATIONS_PATH}/questions.json', 'r') as f:
    questions_data = json.load(f)

with open(f'{ANNOTATIONS_PATH}/exp1_expected_answers.json', 'r') as f:
    expected_answers = json.load(f)

exp1_categories = questions_data['experiment1']['categories']

print('Questions loaded:', list(exp1_categories.keys()))
print('Expected answers loaded for categories:', list(expected_answers.keys()))

In [ ]:
# Inference function
def run_inference(image, question):
    prompt = f'<image>{question}'
    inputs = processor(
        text=prompt,
        images=image,
        padding='longest',
        do_convert_rgb=True,
        return_tensors='pt'
    ).to(device)

    # Match model dtype (important for bfloat16)
    inputs = inputs.to(dtype=model.dtype)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    # Token-based extraction: only decode the generated tokens (after input)
    input_len = inputs['input_ids'].shape[-1]
    generated_ids = output[0][input_len:]
    response = processor.decode(generated_ids, skip_special_tokens=True).strip()
    return response

In [ ]:
# Metrics functions

def normalize(text):
    return str(text).lower().strip()


def calc_exact_match(prediction, expected):
    pred = normalize(prediction)
    exp = normalize(expected)
    pred_tokens = pred.split()
    first_token = pred_tokens[0] if pred_tokens else ''

    if first_token == exp or exp in pred:
        return 1.0

    if exp == 'yes':
        if any(w in pred for w in ['yes', 'there is', 'there are', 'i can see', 'it is', 'this is']):
            return 1.0
    elif exp == 'no':
        if any(w in pred for w in ['no ', 'there is no', 'there are no', 'i cannot', "i don't see", 'not visible', 'no staircase', 'no elevator', 'no door', 'no obstacle']):
            return 1.0
    elif exp in ['open', 'closed', 'up', 'down']:
        if first_token == exp:
            return 1.0

    return 0.0


def calc_keyword_recall(prediction, keywords):
    pred = normalize(prediction)
    for kw in keywords:
        if normalize(kw) in pred:
            return 1.0
    return 0.0


def compute_confusion_matrix(results_df, category, question_id):
    df = results_df[
        (results_df['category'] == category) &
        (results_df['question_id'] == question_id) &
        (results_df['exact_match'].notna())
    ].copy()

    # Use the ground-truth expected answer ("yes"=positive, "no"=negative)
    # This correctly handles cross-category cases like doorsign_bathroom on an exit image
    is_positive_gt = df['expected'].str.lower() == 'yes'

    tp = int(len(df[is_positive_gt & (df['exact_match'] == 1.0)]))
    fn = int(len(df[is_positive_gt & (df['exact_match'] == 0.0)]))
    tn = int(len(df[~is_positive_gt & (df['exact_match'] == 1.0)]))
    fp = int(len(df[~is_positive_gt & (df['exact_match'] == 0.0)]))

    return tp, fp, tn, fn


print('Metrics functions defined')

In [ ]:
# Run Experiment 1 - main inference loop
results = []
total_start = time.time()

for category_name, category_data in exp1_categories.items():
    print(f'\nCategory: {category_name}')
    image_dir = os.path.join(IMAGES_PATH, category_name)

    if not os.path.exists(image_dir):
        print(f'  WARNING: folder not found: {image_dir}')
        continue

    images = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
    category_answers = expected_answers.get(category_name, {})

    for img_name in images:
        is_negative = '_neg_' in img_name
        img_path = os.path.join(image_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        img_answers = category_answers.get(img_name, {})

        if not img_answers:
            print(f'  WARNING: no expected answers for {img_name}')
            continue

        label = 'neg' if is_negative else 'pos'
        print(f'  {img_name} [{label}]')

        for q_data in category_data['questions']:
            q_id = q_data['id']
            question = q_data['question']
            metrics = q_data['metrics']
            q_type = q_data['type']
            expected_key = q_data['expected_answer_key']

            # For negative images: only ask questions that compute tp_fp_tn_fn (detection-like)
            # or element questions. Skip state/detail questions (direction, handrails, open/closed, etc.)
            if is_negative and 'tp_fp_tn_fn' not in metrics and q_type != 'element':
                continue

            expected = img_answers.get(expected_key)
            if expected is None:
                continue

            # Run inference and measure time
            start = time.time()
            prediction = run_inference(image, question)
            inf_time_ms = (time.time() - start) * 1000

            output_length = len(prediction.split())

            row = {
                'model_id': MODEL_ID,
                'category': category_name,
                'image': img_name,
                'is_negative': is_negative,
                'question_id': q_id,
                'question_type': q_type,
                'question': question,
                'prediction': prediction,
                'expected': str(expected),
                'inference_time_ms': round(inf_time_ms, 2),
                'output_length_words': output_length,
                'exact_match': None,
                'keyword_recall': None,
                'bertscore_f1': None,
                'bertscore_p': None,
                'bertscore_r': None,
                'bertscore_reference': None
            }

            if 'exact_match' in metrics:
                row['exact_match'] = calc_exact_match(prediction, str(expected))

            if 'keyword_recall' in metrics:
                keywords = img_answers.get('element_keywords', q_data.get('expected_keywords', []))
                row['keyword_recall'] = calc_keyword_recall(prediction, keywords)

            if 'bertscore' in metrics:
                reference = img_answers.get('bertscore_reference', q_data.get('bertscore_reference', ''))
                if reference:
                    row['bertscore_reference'] = reference

            results.append(row)

total_time_s = time.time() - total_start
print(f'\nTotal inference time: {total_time_s:.1f}s')
print(f'Total rows: {len(results)}')

df = pd.DataFrame(results)
print('DataFrame created')

In [ ]:
# Calculate BERTScore in batch
bertscore_mask = df['bertscore_reference'].notna() & (df['bertscore_reference'] != '')
n_bertscore = bertscore_mask.sum()

if n_bertscore > 0:
    print(f'Calculating BERTScore for {n_bertscore} predictions...')
    predictions_bs = df[bertscore_mask]['prediction'].tolist()
    references_bs = df[bertscore_mask]['bertscore_reference'].tolist()

    P, R, F1 = bert_score_fn(predictions_bs, references_bs, lang='en', verbose=False)

    df.loc[bertscore_mask, 'bertscore_f1'] = F1.numpy()
    df.loc[bertscore_mask, 'bertscore_p'] = P.numpy()
    df.loc[bertscore_mask, 'bertscore_r'] = R.numpy()
    print('BERTScore calculated')
else:
    print('No BERTScore references found')

In [ ]:
# Aggregate metrics per category
print('AGGREGATE METRICS BY CATEGORY')
print('=' * 60)

agg_rows = []

for category_name in exp1_categories.keys():
    cat_df = df[df['category'] == category_name]
    print(f'\n{category_name.upper()}')

    agg = {'category': category_name}

    # Overall accuracy across all exact_match questions
    em_df = cat_df[cat_df['exact_match'].notna()]
    if len(em_df) > 0:
        agg['accuracy'] = round(em_df['exact_match'].mean(), 4)
        print(f'  Accuracy (all closed questions): {agg["accuracy"]:.3f}')

    # Detection confusion matrix
    # door_sign has two detection questions (bathroom + exit) instead of one *_detection
    if category_name == 'door_sign':
        for det_id in ['doorsign_bathroom', 'doorsign_exit']:
            if det_id in cat_df['question_id'].values:
                tp, fp, tn, fn = compute_confusion_matrix(df, category_name, det_id)
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                key = det_id.replace('doorsign_', '')
                agg.update({
                    f'{key}_tp': tp, f'{key}_fp': fp,
                    f'{key}_tn': tn, f'{key}_fn': fn,
                    f'{key}_precision': round(precision, 4),
                    f'{key}_recall': round(recall, 4),
                    f'{key}_f1': round(f1, 4)
                })
                print(f'  {det_id}  TP:{tp} FP:{fp} TN:{tn} FN:{fn}')
                print(f'  {det_id}  Precision:{precision:.3f}  Recall:{recall:.3f}  F1:{f1:.3f}')
    else:
        det_id = f'{category_name}_detection'
        if det_id in cat_df['question_id'].values:
            tp, fp, tn, fn = compute_confusion_matrix(df, category_name, det_id)
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

            agg.update({
                'detection_tp': tp, 'detection_fp': fp,
                'detection_tn': tn, 'detection_fn': fn,
                'detection_precision': round(precision, 4),
                'detection_recall': round(recall, 4),
                'detection_f1': round(f1, 4)
            })
            print(f'  Detection  TP:{tp} FP:{fp} TN:{tn} FN:{fn}')
            print(f'  Detection  Precision:{precision:.3f}  Recall:{recall:.3f}  F1:{f1:.3f}')

            if category_name == 'obstacle':
                hmr = round(1 - recall, 4)
                agg['hazard_miss_rate'] = hmr
                print(f'  Hazard Miss Rate (detection): {hmr:.3f}')

                # Obstacle safety: also a safety-critical detection question
                if 'obstacle_safety' in cat_df['question_id'].values:
                    tp_s, fp_s, tn_s, fn_s = compute_confusion_matrix(df, category_name, 'obstacle_safety')
                    prec_s = tp_s / (tp_s + fp_s) if (tp_s + fp_s) > 0 else 0.0
                    rec_s = tp_s / (tp_s + fn_s) if (tp_s + fn_s) > 0 else 0.0
                    f1_s = 2 * prec_s * rec_s / (prec_s + rec_s) if (prec_s + rec_s) > 0 else 0.0
                        safety_hmr = round(fp_s / (fp_s + tn_s), 4) if (fp_s + tn_s) > 0 else 0.0
                        agg.update({
                            'safety_tp': tp_s, 'safety_fp': fp_s,
                            'safety_tn': tn_s, 'safety_fn': fn_s,
                            'safety_precision': round(prec_s, 4),
                            'safety_recall': round(rec_s, 4),
                            'safety_f1': round(f1_s, 4),
                            'safety_hazard_miss_rate': safety_hmr
                        })
                        print(f'  Safety     TP:{tp_s} FP:{fp_s} TN:{tn_s} FN:{fn_s}')
                        print(f'  Safety     Precision:{prec_s:.3f}  Recall:{rec_s:.3f}  F1:{f1_s:.3f}')
                        print(f'  Safety Hazard Miss Rate: {safety_hmr:.3f} (FP/(FP+TN))')

    # Keyword recall
    kr_df = cat_df[cat_df['keyword_recall'].notna()]
    if len(kr_df) > 0:
        agg['keyword_recall'] = round(kr_df['keyword_recall'].mean(), 4)
        print(f'  Keyword Recall (element): {agg["keyword_recall"]:.3f}')

    # BERTScore
    bs_df = cat_df[cat_df['bertscore_f1'].notna()]
    if len(bs_df) > 0:
        agg['bertscore_f1'] = round(bs_df['bertscore_f1'].mean(), 4)
        agg['bertscore_p'] = round(bs_df['bertscore_p'].mean(), 4)
        agg['bertscore_r'] = round(bs_df['bertscore_r'].mean(), 4)
        print(f'  BERTScore F1 (element): {agg["bertscore_f1"]:.3f}')

    # Output length
    agg['avg_output_length'] = round(cat_df['output_length_words'].mean(), 2)
    print(f'  Avg output length: {agg["avg_output_length"]} words')

    agg_rows.append(agg)

agg_df = pd.DataFrame(agg_rows)
print('\nSUMMARY TABLE')
display(agg_df)

In [ ]:
# Per-image detailed results
print('PER-IMAGE RESULTS')
print('=' * 60)
display_cols = ['category', 'image', 'question_id', 'prediction', 'expected',
                'exact_match', 'keyword_recall', 'bertscore_f1']
display(df[display_cols])

In [ ]:
# Efficiency metrics
vram_peak_gb = torch.cuda.max_memory_allocated() / (1024**3) if device == 'cuda' else 0
n_images = df['image'].nunique()
total_inf_ms = df['inference_time_ms'].sum()

efficiency = {
    'model_id': MODEL_ID,
    'experiment': 'exp1',
    'n_parameters': n_parameters,
    'n_parameters_billions': round(n_parameters / 1e9, 2),
    'n_images': n_images,
    'n_inferences': len(df),
    'vram_peak_gb': round(vram_peak_gb, 3),
    'model_load_time_s': model_load_time_s,
    'total_wall_time_s': round(total_time_s, 2),
    'total_inference_time_s': round(total_inf_ms / 1000, 2),
    'avg_inference_time_ms': round(df['inference_time_ms'].mean(), 2),
    'throughput_img_per_s': round(n_images / (total_inf_ms / 1000), 3) if total_inf_ms > 0 else 0,
    'throughput_inf_per_s': round(len(df) / (total_inf_ms / 1000), 3) if total_inf_ms > 0 else 0,
    'avg_output_length_words': round(df['output_length_words'].mean(), 2),
    'max_output_length_words': int(df['output_length_words'].max())
}

print('EFFICIENCY METRICS')
print('=' * 40)
for k, v in efficiency.items():
    print(f'  {k}: {v}')

In [ ]:
# Save results to CSV and JSON
os.makedirs(RESULTS_PATH, exist_ok=True)
model_name_safe = MODEL_ID.replace('/', '_').replace('-', '_')

results_file = f'{RESULTS_PATH}/{model_name_safe}_exp1_results.csv'
agg_file = f'{RESULTS_PATH}/{model_name_safe}_exp1_aggregate.csv'
efficiency_file = f'{RESULTS_PATH}/{model_name_safe}_exp1_efficiency.json'

df.to_csv(results_file, index=False)
agg_df.to_csv(agg_file, index=False)

with open(efficiency_file, 'w') as f:
    json.dump(efficiency, f, indent=2)

print(f'Results saved to: {results_file}')
print(f'Aggregate saved to: {agg_file}')
print(f'Efficiency saved to: {efficiency_file}')

In [ ]:
# State confusion matrices
# open/closed: elevator_doors, door_state, doorsign_state
# up/down: staircase_direction

def build_state_confusion(results_df, question_ids, labels):
    """
    Builds a confusion matrix for state questions with fixed label set.
    Returns (matrix, accuracy, total) where matrix[true_idx][pred_idx].
    Unknown predictions (not in labels) are counted in totals but not placed in matrix.
    """
    subset = results_df[results_df['question_id'].isin(question_ids)].copy()
    subset = subset[subset['prediction'].notna() & subset['expected'].notna()]

    label_to_idx = {l: i for i, l in enumerate(labels)}
    n = len(labels)
    matrix = [[0] * n for _ in range(n)]
    correct, total, unknown_pred = 0, 0, 0

    for _, row in subset.iterrows():
        pred_tokens = normalize(str(row['prediction'])).split()
        pred = pred_tokens[0] if pred_tokens else ''
        exp = normalize(str(row['expected']))

        pi = label_to_idx.get(pred)
        ei = label_to_idx.get(exp)

        if ei is None:
            continue  # skip rows where expected is not a valid label

        total += 1
        if pi is not None:
            matrix[ei][pi] += 1
            if pi == ei:
                correct += 1
        else:
            unknown_pred += 1  # model output didn't match any known label

    accuracy = correct / total if total > 0 else 0.0
    return matrix, correct, total, unknown_pred


# ── Open / Closed ──────────────────────────────────────────────
oc_question_ids = ['elevator_doors', 'door_state', 'doorsign_state']
oc_labels = ['open', 'closed']
oc_matrix, oc_correct, oc_total, oc_unknown = build_state_confusion(df, oc_question_ids, oc_labels)
oc_accuracy = oc_correct / oc_total if oc_total > 0 else 0.0

print('STATE CONFUSION MATRIX: open / closed')
print('(rows = ground truth, columns = predicted)')
print(f"{'':>10} {'open':>8} {'closed':>8}")
for i, label in enumerate(oc_labels):
    print(f"{label:>10} {oc_matrix[i][0]:>8} {oc_matrix[i][1]:>8}")
print(f"Accuracy: {oc_accuracy:.3f}  ({oc_correct}/{oc_total})  Unknown predictions: {oc_unknown}")

# ── Up / Down ──────────────────────────────────────────────────
ud_question_ids = ['staircase_direction']
ud_labels = ['up', 'down']
ud_matrix, ud_correct, ud_total, ud_unknown = build_state_confusion(df, ud_question_ids, ud_labels)
ud_accuracy = ud_correct / ud_total if ud_total > 0 else 0.0

print('\nSTATE CONFUSION MATRIX: up / down')
print('(rows = ground truth, columns = predicted)')
print(f"{'':>10} {'up':>8} {'down':>8}")
for i, label in enumerate(ud_labels):
    print(f"{label:>10} {ud_matrix[i][0]:>8} {ud_matrix[i][1]:>8}")
print(f"Accuracy: {ud_accuracy:.3f}  ({ud_correct}/{ud_total})  Unknown predictions: {ud_unknown}")

# ── Per-question-id breakdown for open/closed ──────────────────
print('\nOpen/Closed breakdown by question_id:')
for qid in oc_question_ids:
    sub = df[(df['question_id'] == qid) & df['exact_match'].notna()]
    if len(sub) > 0:
        acc = sub['exact_match'].mean()
        print(f"  {qid}: accuracy={acc:.3f} (n={len(sub)})")

In [ ]:
# Navigation elements confusion matrix
# Evaluates element/object questions: "What navigation element is visible?"
# Maps free-text predictions to canonical categories.

ELEMENT_CANONICAL = {
    'staircase': ['stair', 'stairs', 'staircase', 'stairway', 'steps'],
    'elevator': ['elevator', 'lift'],
    'door': ['door', 'doorway', 'entrance'],
    # door_sign model responds "bathroom"/"exit"/"restroom" — all valid sign descriptions
    'sign': ['sign', 'signal', 'notice', 'label', 'placard', 'plaque', 'board',
             'bathroom', 'restroom', 'toilet', 'wc', 'exit', 'emergency'],
    'obstacle': ['obstacle', 'object', 'obstruction', 'barrier', 'block', 'box', 'cart',
                 'chair', 'table', 'rack', 'shelf', 'shelves', 'trolley', 'furniture',
                 'plant', 'bin', 'bucket', 'bag', 'luggage'],
    'corridor': ['corridor', 'hallway', 'hall', 'passage', 'aisle'],
    'other': ['other', 'none', 'unknown']
}

# Ground-truth category → canonical element label
GT_ELEMENT_MAP = {
    'staircase': 'staircase',
    'elevator': 'elevator',
    'door': 'door',
    'door_sign': 'sign',
    'obstacle': 'obstacle'
}

# Element question IDs per category (obstacle uses 'obstacle_object', others use '*_element')
ELEMENT_QUESTION_IDS = {
    'staircase': 'staircase_element',
    'elevator': 'elevator_element',
    'door': 'door_element',
    'door_sign': 'doorsign_element',
    'obstacle': 'obstacle_object'
}

def pred_to_canonical(prediction):
    """Map a free-text prediction to its canonical element label, or 'other'."""
    pred = normalize(str(prediction))
    for canon, keywords in ELEMENT_CANONICAL.items():
        if any(kw in pred for kw in keywords):
            return canon
    return 'other'

# Build the matrix — only positive images, one element question per category
el_labels = ['staircase', 'elevator', 'door', 'sign', 'obstacle', 'corridor', 'other']
el_label_to_idx = {l: i for i, l in enumerate(el_labels)}
n = len(el_labels)
el_matrix = [[0] * n for _ in range(n)]
el_correct, el_total = 0, 0

valid_qids = set(ELEMENT_QUESTION_IDS.values())
el_df = df[(df['question_id'].isin(valid_qids)) & (~df['is_negative'])].copy()

for _, row in el_df.iterrows():
    gt_canon = GT_ELEMENT_MAP.get(row['category'])
    if gt_canon is None:
        continue
    pred_canon = pred_to_canonical(row['prediction'])

    ei = el_label_to_idx.get(gt_canon)
    pi = el_label_to_idx.get(pred_canon)
    if ei is None or pi is None:
        continue

    el_matrix[ei][pi] += 1
    el_total += 1
    if ei == pi:
        el_correct += 1

el_accuracy = el_correct / el_total if el_total > 0 else 0.0

# Print matrix
print('NAVIGATION ELEMENTS CONFUSION MATRIX')
print('(rows = ground truth, columns = predicted)')
col_w = 11
header = f"{'':>{col_w}}" + ''.join(f"{l:>{col_w}}" for l in el_labels)
print(header)
for i, label in enumerate(el_labels):
    row_str = f"{label:>{col_w}}" + ''.join(f"{el_matrix[i][j]:>{col_w}}" for j in range(n))
    print(row_str)
print(f"\nElement accuracy: {el_accuracy:.3f}  ({el_correct}/{el_total})")

# Per-category breakdown
print('\nPer-category element recognition:')
for cat, qid in ELEMENT_QUESTION_IDS.items():
    sub = el_df[el_df['category'] == cat]
    if len(sub) == 0:
        print(f"  {cat}: no data")
        continue
    gt_label = GT_ELEMENT_MAP[cat]
    cat_correct = sum(pred_to_canonical(p) == gt_label for p in sub['prediction'])
    print(f"  {cat} ({qid}): {cat_correct}/{len(sub)} = {cat_correct/len(sub):.3f}")
    # Show what the model actually predicts for mismatches
    for _, r in sub[sub['prediction'].notna()].iterrows():
        mapped = pred_to_canonical(r['prediction'])
        if mapped != gt_label:
            print(f"    ✗ pred='{r['prediction']}' → mapped='{mapped}' (expected '{gt_label}')")

In [ ]:
# Save confusion matrices to JSON
matrices = {
    'model_id': MODEL_ID,
    'experiment': 'exp1',
    'state_open_closed': {
        'labels': oc_labels,
        'matrix': oc_matrix,
        'accuracy': round(oc_accuracy, 4),
        'correct': oc_correct,
        'total': oc_total,
        'unknown_predictions': oc_unknown,
        'question_ids': oc_question_ids,
        'note': 'Rows = ground truth, columns = predicted'
    },
    'state_up_down': {
        'labels': ud_labels,
        'matrix': ud_matrix,
        'accuracy': round(ud_accuracy, 4),
        'correct': ud_correct,
        'total': ud_total,
        'unknown_predictions': ud_unknown,
        'question_ids': ud_question_ids,
        'note': 'Rows = ground truth, columns = predicted'
    },
    'navigation_elements': {
        'labels': el_labels,
        'matrix': el_matrix,
        'accuracy': round(el_accuracy, 4),
        'correct': el_correct,
        'total': el_total,
        'gt_element_map': GT_ELEMENT_MAP,
        'note': 'Rows = ground truth category, columns = predicted canonical element'
    }
}

matrices_file = f'{RESULTS_PATH}/{model_name_safe}_exp1_matrices.json'
with open(matrices_file, 'w') as f:
    json.dump(matrices, f, indent=2)

print(f'Matrices saved to: {matrices_file}')
print('\nAll experiment 1 results saved:')
print(f'  {results_file}')
print(f'  {agg_file}')
print(f'  {efficiency_file}')
print(f'  {matrices_file}')
print('\nDownload /content/results/exp1/ as ZIP when done.')

In [ ]:
# Generate and save confusion matrix plots as PNG
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plots_path = f'{RESULTS_PATH}/plots'
os.makedirs(plots_path, exist_ok=True)

def save_heatmap(matrix, labels_row, labels_col, title, filename, fmt='d', cmap='Blues', annot_fontsize=14):
    """Save a confusion matrix as a heatmap PNG."""
    matrix_np = np.array(matrix)
    fig, ax = plt.subplots(figsize=(max(5, len(labels_col)*1.4), max(4, len(labels_row)*1.2)))
    im = ax.imshow(matrix_np, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(labels_col)))
    ax.set_yticks(range(len(labels_row)))
    ax.set_xticklabels(labels_col, fontsize=11)
    ax.set_yticklabels(labels_row, fontsize=11)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Ground Truth', fontsize=12)
    ax.set_title(f'{title}\n{MODEL_ID}', fontsize=12, pad=10)

    # Annotate cells
    thresh = matrix_np.max() / 2.0
    for i in range(len(labels_row)):
        for j in range(len(labels_col)):
            val = matrix_np[i, j]
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=annot_fontsize,
                    color='white' if val > thresh else 'black')

    plt.tight_layout()
    fpath = f'{plots_path}/{filename}'
    plt.savefig(fpath, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {fpath}')


# ── 1. State: open / closed ────────────────────────────────────
save_heatmap(
    oc_matrix, oc_labels, oc_labels,
    'State Recognition: Open / Closed',
    f'{model_name_safe}_state_open_closed.png'
)

# ── 2. State: up / down ───────────────────────────────────────
save_heatmap(
    ud_matrix, ud_labels, ud_labels,
    'State Recognition: Up / Down (Staircase Direction)',
    f'{model_name_safe}_state_up_down.png'
)

# ── 3. Navigation elements ────────────────────────────────────
save_heatmap(
    el_matrix, el_labels, el_labels,
    'Navigation Elements Recognition',
    f'{model_name_safe}_nav_elements.png',
    annot_fontsize=12
)

# ── 4. Detection 2×2 per category ─────────────────────────────
detection_cats = {
    'staircase': ('staircase_detection', 'Staircase Detection'),
    'elevator':  ('elevator_detection',  'Elevator Detection'),
    'door':      ('door_detection',      'Door Detection'),
    'obstacle':  ('obstacle_detection',  'Obstacle Detection'),
}

for cat_name, (q_id, title) in detection_cats.items():
    tp, fp, tn, fn = compute_confusion_matrix(df, cat_name, q_id)
    mat = [[tp, fn], [fp, tn]]   # rows: actual pos/neg, cols: pred pos/neg
    save_heatmap(
        mat,
        labels_row=['Actual Positive', 'Actual Negative'],
        labels_col=['Pred Positive', 'Pred Negative'],
        title=title,
        filename=f'{model_name_safe}_detection_{cat_name}.png',
        cmap='Greens'
    )

# door_sign: bathroom + exit separately
for det_id, title in [('doorsign_bathroom', 'Door Sign: Bathroom Detection'),
                       ('doorsign_exit',     'Door Sign: Exit Detection')]:
    tp, fp, tn, fn = compute_confusion_matrix(df, 'door_sign', det_id)
    mat = [[tp, fn], [fp, tn]]
    save_heatmap(
        mat,
        labels_row=['Actual Positive', 'Actual Negative'],
        labels_col=['Pred Positive', 'Pred Negative'],
        title=title,
        filename=f'{model_name_safe}_detection_{det_id}.png',
        cmap='Greens'
    )

# ── 5. Obstacle safety 2×2 ────────────────────────────────────
tp_s, fp_s, tn_s, fn_s = compute_confusion_matrix(df, 'obstacle', 'obstacle_safety')
mat_s = [[tp_s, fn_s], [fp_s, tn_s]]
save_heatmap(
    mat_s,
    labels_row=['Actual Safe', 'Actual Unsafe'],
    labels_col=['Pred Safe', 'Pred Unsafe'],
    title='Obstacle Safety Classification',
    filename=f'{model_name_safe}_safety_obstacle.png',
    cmap='Oranges'
)

# ── 6. Per-category accuracy bar chart ───────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
cats = agg_df['category'].tolist()
accs = agg_df['accuracy'].tolist()
colors = ['#4CAF50' if a >= 0.9 else '#FF9800' if a >= 0.7 else '#F44336' for a in accs]
bars = ax.bar(cats, accs, color=colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title(f'Accuracy by Category\n{MODEL_ID}', fontsize=12)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=10)
patches = [mpatches.Patch(color='#4CAF50', label='≥ 0.9'),
           mpatches.Patch(color='#FF9800', label='0.7–0.9'),
           mpatches.Patch(color='#F44336', label='< 0.7')]
ax.legend(handles=patches, fontsize=9)
plt.tight_layout()
bar_path = f'{plots_path}/{model_name_safe}_accuracy_by_category.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {bar_path}')

print(f'\nAll plots saved to: {plots_path}')
print('Download /content/results/exp1/ as ZIP to get CSVs, JSONs and PNGs.')